# Chapter 2.5: Attention-based CTR Models

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why **user behavior sequences** are critical for CTR prediction
2. Implement the **DIN** (Deep Interest Network, Alibaba 2018) attention mechanism
3. Understand **DIEN** (Deep Interest Evolution Network): GRU + attention for interest evolution
4. Describe **DSIN** (Deep Session Interest Network): session-level behavior modeling
5. Implement **BST** (Behavior Sequence Transformer, Alibaba 2019): Transformer encoder for behavior
6. Compare fixed-pooling vs attention-based behavior modeling
7. Visualize attention weights to interpret user interest patterns

## Prerequisites

- Chapters 2.1-2.4 (CTR prediction fundamentals)
- Understanding of attention mechanisms and Transformers
- PyTorch experience with sequence models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part2/chapter_2.5_attention_ctr.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part2/chapter_2.5_attention_ctr.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cpu')

## 1. Why User Behavior Sequences Matter

Traditional CTR models (LR, FM, DeepFM) treat user features as **static** profiles. But user interests are **dynamic** and **diverse**:

- A user who browsed electronics, books, and clothing has diverse interests
- When predicting CTR for a phone ad, the electronics browsing history is more relevant
- Interest evolves over time: recent behaviors may be more important

The key insight of DIN is: **not all behaviors are equally relevant to the target item**.

Standard approach (sum/mean pooling):
$$ \mathbf{v}_U = \frac{1}{|S|} \sum_{i \in S} \mathbf{e}_i $$

DIN approach (attention-weighted):
$$ \mathbf{v}_U = \sum_{i \in S} \alpha(\mathbf{e}_i, \mathbf{e}_t) \cdot \mathbf{e}_i $$

where $\alpha(\mathbf{e}_i, \mathbf{e}_t)$ measures the relevance of behavior item $i$ to target item $t$.

## 2. Generating Synthetic Behavior Sequence Data

In [ ]:
def generate_behavior_data(n_users=5000, n_items=200, n_categories=10,
                           max_seq_len=20, seed=42):
    """
    Generate synthetic CTR data with user behavior sequences.
    Each sample: (user_profile, behavior_sequence, target_item) -> click/no-click
    """
    rng = np.random.RandomState(seed)
    
    # Item-to-category mapping
    item_category = rng.randint(0, n_categories, n_items)
    
    # User category preferences (latent)
    user_pref = rng.randn(n_users, n_categories) * 0.5
    
    # Item embeddings (latent)
    item_embed = rng.randn(n_items, 8) * 0.3
    
    samples = []
    for _ in range(n_users * 4):  # ~4 samples per user
        uid = rng.randint(0, n_users)
        
        # Generate behavior sequence
        seq_len = rng.randint(3, max_seq_len + 1)
        # Behaviors biased by user preference
        cat_probs = np.exp(user_pref[uid]) / np.sum(np.exp(user_pref[uid]))
        behavior_cats = rng.choice(n_categories, seq_len, p=cat_probs)
        behavior_items = []
        for c in behavior_cats:
            items_in_cat = np.where(item_category == c)[0]
            behavior_items.append(rng.choice(items_in_cat))
        
        # Target item
        target_item = rng.randint(0, n_items)
        target_cat = item_category[target_item]
        
        # Click probability: higher if user has browsed similar category
        category_overlap = np.sum(behavior_cats == target_cat)
        # Also depends on embedding similarity
        sim = np.mean([np.dot(item_embed[b], item_embed[target_item]) 
                       for b in behavior_items])
        
        logit = -1.5 + 0.5 * category_overlap + 2.0 * sim + user_pref[uid, target_cat]
        prob = 1.0 / (1.0 + np.exp(-logit))
        label = rng.binomial(1, np.clip(prob, 0.01, 0.99))
        
        # Pad behavior sequence
        padded_seq = np.zeros(max_seq_len, dtype=np.int64)
        padded_seq[:seq_len] = behavior_items
        mask = np.zeros(max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        
        samples.append({
            'user_id': uid,
            'behavior_seq': padded_seq,
            'mask': mask,
            'target_item': target_item,
            'label': label,
            'seq_len': seq_len
        })
    
    # Convert to arrays
    behavior_seqs = np.array([s['behavior_seq'] for s in samples])
    masks = np.array([s['mask'] for s in samples])
    target_items = np.array([s['target_item'] for s in samples])
    labels = np.array([s['label'] for s in samples])
    
    print(f"Generated {len(samples)} samples, CTR={labels.mean():.4f}")
    print(f"Items: {n_items}, Categories: {n_categories}, Max seq len: {max_seq_len}")
    
    return behavior_seqs, masks, target_items, labels, n_items, item_category

behavior_seqs, masks, target_items, labels, n_items, item_category = generate_behavior_data()

# Split
split = 16000
train_seq, test_seq = behavior_seqs[:split], behavior_seqs[split:]
train_mask, test_mask = masks[:split], masks[split:]
train_target, test_target = target_items[:split], target_items[split:]
train_labels, test_labels = labels[:split], labels[split:]

## 3. Baseline: Mean Pooling Model

First, let's build a baseline that simply averages all behavior embeddings.

In [ ]:
class MeanPoolingCTR(nn.Module):
    """Baseline CTR model with mean pooling of behavior sequence."""
    
    def __init__(self, n_items, embed_dim=16, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 32]
        
        self.item_embedding = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        self.item_embedding.weight.data[0] = 0  # Padding vector
        
        # MLP: concat(user_behavior, target_embed) -> prediction
        input_dim = embed_dim * 2
        layers = []
        for h in hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, behavior_seq, mask, target_item):
        # Shift item IDs by 1 to reserve 0 for padding
        seq_embed = self.item_embedding(behavior_seq + 1)  # (batch, seq_len, d)
        target_embed = self.item_embedding(target_item + 1)  # (batch, d)
        
        # Mean pooling with mask
        mask_expanded = mask.unsqueeze(-1)  # (batch, seq_len, 1)
        seq_sum = (seq_embed * mask_expanded).sum(dim=1)  # (batch, d)
        seq_len = mask.sum(dim=1, keepdim=True).clamp(min=1)  # (batch, 1)
        user_embed = seq_sum / seq_len  # (batch, d)
        
        # Concat and predict
        combined = torch.cat([user_embed, target_embed], dim=1)
        return self.mlp(combined).squeeze(-1)

print("MeanPooling model defined.")

## 4. DIN: Deep Interest Network (Alibaba, 2018)

DIN's key innovation is **target-aware attention**: the attention weight of each behavior depends on its relevance to the current target item.

The attention function:

$$ \alpha_i = \frac{\exp(a(\mathbf{e}_i, \mathbf{e}_t))}{\sum_{j \in S} \exp(a(\mathbf{e}_j, \mathbf{e}_t))} $$

where $a(\cdot, \cdot)$ is a learned attention network:

$$ a(\mathbf{e}_i, \mathbf{e}_t) = \text{MLP}([\mathbf{e}_i, \mathbf{e}_t, \mathbf{e}_i - \mathbf{e}_t, \mathbf{e}_i \odot \mathbf{e}_t]) $$

> **Concept:** DIN does NOT use softmax normalization in practice -- it uses raw attention scores. This allows the model to express "no relevant history" with all-low scores, rather than forcing a distribution that always sums to 1.

In [ ]:
class DINAttention(nn.Module):
    """DIN attention unit: target-aware attention over behavior sequence."""
    
    def __init__(self, embed_dim, attention_hidden=32):
        super().__init__()
        # Input: [e_i, e_t, e_i - e_t, e_i * e_t]
        input_dim = embed_dim * 4
        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_hidden),
            nn.ReLU(),
            nn.Linear(attention_hidden, 1)
        )
    
    def forward(self, behavior_embed, target_embed, mask):
        """
        behavior_embed: (batch, seq_len, d)
        target_embed: (batch, d)
        mask: (batch, seq_len)
        Returns: weighted behavior representation (batch, d)
        """
        seq_len = behavior_embed.size(1)
        target_expanded = target_embed.unsqueeze(1).expand_as(behavior_embed)  # (batch, seq, d)
        
        # Attention input: [e_i, e_t, e_i - e_t, e_i * e_t]
        att_input = torch.cat([
            behavior_embed,
            target_expanded,
            behavior_embed - target_expanded,
            behavior_embed * target_expanded
        ], dim=-1)  # (batch, seq, 4d)
        
        # Compute attention scores
        scores = self.attention(att_input).squeeze(-1)  # (batch, seq)
        
        # Apply mask (set padding positions to large negative)
        scores = scores * mask + (1 - mask) * (-1e9)
        
        # DIN uses softmax (or raw scores in some implementations)
        weights = torch.softmax(scores, dim=1)  # (batch, seq)
        weights = weights * mask  # Zero out padding
        
        # Weighted sum
        output = (behavior_embed * weights.unsqueeze(-1)).sum(dim=1)  # (batch, d)
        
        return output, weights


class DIN(nn.Module):
    """Deep Interest Network (Alibaba, 2018)."""
    
    def __init__(self, n_items, embed_dim=16, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 32]
        
        self.item_embedding = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        self.item_embedding.weight.data[0] = 0
        
        self.attention = DINAttention(embed_dim)
        
        # MLP
        input_dim = embed_dim * 2
        layers = []
        for h in hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, behavior_seq, mask, target_item):
        seq_embed = self.item_embedding(behavior_seq + 1)
        target_embed = self.item_embedding(target_item + 1)
        
        # Attention-weighted behavior representation
        user_embed, self.last_attention_weights = self.attention(
            seq_embed, target_embed, mask)
        
        combined = torch.cat([user_embed, target_embed], dim=1)
        return self.mlp(combined).squeeze(-1)

print("DIN model defined.")

## 5. BST: Behavior Sequence Transformer (Alibaba, 2019)

BST applies the **Transformer encoder** to model behavior sequences, capturing:
- **Self-attention** between behavior items (not just target-aware)
- **Positional encoding** for sequence order
- Multi-head attention for diverse interest patterns

In [ ]:
class BST(nn.Module):
    """Behavior Sequence Transformer (Alibaba, 2019)."""
    
    def __init__(self, n_items, embed_dim=16, n_heads=2, n_layers=1,
                 max_seq_len=20, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 32]
        
        self.embed_dim = embed_dim
        
        self.item_embedding = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        self.item_embedding.weight.data[0] = 0
        
        # Positional encoding
        self.position_embedding = nn.Embedding(max_seq_len + 1, embed_dim)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, 
            dim_feedforward=embed_dim * 4, dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # MLP
        input_dim = embed_dim * 2  # transformer output + target
        layers = []
        for h in hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, behavior_seq, mask, target_item):
        batch_size, seq_len = behavior_seq.shape
        
        # Item embeddings + positional embeddings
        seq_embed = self.item_embedding(behavior_seq + 1)  # (batch, seq, d)
        positions = torch.arange(seq_len, device=behavior_seq.device).unsqueeze(0).expand(batch_size, -1)
        pos_embed = self.position_embedding(positions)
        seq_embed = seq_embed + pos_embed
        
        # Create attention mask for padding (True = ignore)
        src_key_padding_mask = (mask == 0)  # (batch, seq)
        
        # Transformer encoding
        transformer_out = self.transformer(seq_embed, 
                                           src_key_padding_mask=src_key_padding_mask)
        
        # Mean pooling over valid positions
        mask_expanded = mask.unsqueeze(-1)
        user_embed = (transformer_out * mask_expanded).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1)
        
        target_embed = self.item_embedding(target_item + 1)
        combined = torch.cat([user_embed, target_embed], dim=1)
        return self.mlp(combined).squeeze(-1)

print("BST model defined.")

## 6. Training and Comparison

In [ ]:
def train_seq_model(model, train_data, test_data, epochs=20, batch_size=256, lr=0.001):
    """Train a sequence-based CTR model."""
    train_seq, train_mask, train_target, train_y = train_data
    test_seq, test_mask, test_target, test_y = test_data
    
    train_ds = TensorDataset(
        torch.LongTensor(train_seq), torch.FloatTensor(train_mask),
        torch.LongTensor(train_target), torch.FloatTensor(train_y)
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    
    history = {'train_loss': [], 'test_auc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        for b_seq, b_mask, b_target, b_y in loader:
            optimizer.zero_grad()
            logits = model(b_seq, b_mask, b_target)
            loss = criterion(logits, b_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        
        model.eval()
        with torch.no_grad():
            t_logits = model(
                torch.LongTensor(test_seq),
                torch.FloatTensor(test_mask),
                torch.LongTensor(test_target)
            )
            t_probs = torch.sigmoid(t_logits).numpy()
            auc = roc_auc_score(test_y, t_probs)
        
        history['train_loss'].append(total_loss / n_batches)
        history['test_auc'].append(auc)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: loss={total_loss/n_batches:.4f}, AUC={auc:.4f}")
    
    return history

train_data = (train_seq, train_mask, train_target, train_labels)
test_data = (test_seq, test_mask, test_target, test_labels)

# Mean Pooling Baseline
print("=== Mean Pooling ===")
torch.manual_seed(42)
mean_model = MeanPoolingCTR(n_items, embed_dim=16)
mean_hist = train_seq_model(mean_model, train_data, test_data)

# DIN
print("\n=== DIN ===")
torch.manual_seed(42)
din_model = DIN(n_items, embed_dim=16)
din_hist = train_seq_model(din_model, train_data, test_data)

# BST
print("\n=== BST ===")
torch.manual_seed(42)
bst_model = BST(n_items, embed_dim=16, n_heads=2, n_layers=1)
bst_hist = train_seq_model(bst_model, train_data, test_data)

In [ ]:
# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, 21)

axes[0].plot(epochs, mean_hist['train_loss'], 'b-', label='Mean Pooling', linewidth=2)
axes[0].plot(epochs, din_hist['train_loss'], 'r-', label='DIN', linewidth=2)
axes[0].plot(epochs, bst_hist['train_loss'], 'g-', label='BST', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, mean_hist['test_auc'], 'b-o', label='Mean Pooling', linewidth=2, markersize=3)
axes[1].plot(epochs, din_hist['test_auc'], 'r-s', label='DIN', linewidth=2, markersize=3)
axes[1].plot(epochs, bst_hist['test_auc'], 'g-^', label='BST', linewidth=2, markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test AUC')
axes[1].set_title('Test AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final AUC -- Mean: {mean_hist['test_auc'][-1]:.4f}, "
      f"DIN: {din_hist['test_auc'][-1]:.4f}, "
      f"BST: {bst_hist['test_auc'][-1]:.4f}")

## 7. Visualizing DIN Attention Weights

In [ ]:
# Visualize attention for a few test samples
din_model.eval()
with torch.no_grad():
    sample_idx = [0, 1, 2, 3]
    s_seq = torch.LongTensor(test_seq[sample_idx])
    s_mask = torch.FloatTensor(test_mask[sample_idx])
    s_target = torch.LongTensor(test_target[sample_idx])
    
    logits = din_model(s_seq, s_mask, s_target)
    probs = torch.sigmoid(logits).numpy()
    att_weights = din_model.last_attention_weights.numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, ax in enumerate(axes.flat):
    seq_len = int(test_mask[sample_idx[idx]].sum())
    weights = att_weights[idx, :seq_len]
    behavior_items = test_seq[sample_idx[idx], :seq_len]
    target = test_target[sample_idx[idx]]
    
    # Color by category match
    target_cat = item_category[target]
    colors = ['red' if item_category[b] == target_cat else 'steelblue' 
              for b in behavior_items]
    
    bars = ax.bar(range(seq_len), weights, color=colors, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Behavior Position')
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'Sample {idx+1}: Target cat={target_cat}, P(click)={probs[idx]:.3f}\n'
                 f'Red=same category as target')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. DIEN: Deep Interest Evolution Network (Alibaba, 2019)

DIEN models **interest evolution** with two key innovations:

1. **Interest Extractor**: GRU to extract sequential interest states
2. **Interest Evolution**: AUGRU (Attention-based GRU) to model how interests evolve w.r.t. the target

$$ \tilde{\mathbf{h}}_t = \alpha_t \cdot \mathbf{h}'_t \quad \text{(attention-modulated hidden state)} $$

> **Concept:** While DIN treats behavior as a set (no order), DIEN explicitly models the temporal evolution of user interests. Recent behaviors that are relevant to the target item get higher importance.

In [ ]:
class DIEN(nn.Module):
    """Simplified DIEN: GRU + target-aware attention (Alibaba, 2019)."""
    
    def __init__(self, n_items, embed_dim=16, gru_hidden=16, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 32]
        
        self.item_embedding = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        self.item_embedding.weight.data[0] = 0
        
        # Interest Extractor: GRU
        self.gru = nn.GRU(embed_dim, gru_hidden, batch_first=True)
        
        # Attention over GRU hidden states
        self.attention = DINAttention(gru_hidden)
        
        # MLP
        input_dim = gru_hidden + embed_dim
        layers = []
        for h in hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, behavior_seq, mask, target_item):
        seq_embed = self.item_embedding(behavior_seq + 1)
        target_embed = self.item_embedding(target_item + 1)
        
        # GRU to extract interest states
        gru_out, _ = self.gru(seq_embed)  # (batch, seq, gru_hidden)
        
        # Attention over GRU hidden states, target-aware
        # Project target to GRU hidden size for attention
        user_embed, _ = self.attention(gru_out, 
                                       target_embed[:, :gru_out.size(-1)],  # truncate if needed
                                       mask)
        
        combined = torch.cat([user_embed, target_embed], dim=1)
        return self.mlp(combined).squeeze(-1)

print("\n=== DIEN ===")
torch.manual_seed(42)
dien_model = DIEN(n_items, embed_dim=16, gru_hidden=16)
dien_hist = train_seq_model(dien_model, train_data, test_data)

print(f"\nFinal AUC comparison:")
print(f"  Mean Pooling: {mean_hist['test_auc'][-1]:.4f}")
print(f"  DIN:          {din_hist['test_auc'][-1]:.4f}")
print(f"  DIEN:         {dien_hist['test_auc'][-1]:.4f}")
print(f"  BST:          {bst_hist['test_auc'][-1]:.4f}")

## 9. DSIN: Deep Session Interest Network (Alibaba, 2019)

DSIN observes that user behaviors are naturally **grouped into sessions**. Within a session, behaviors are highly correlated; across sessions, interests may shift.

DSIN models:
1. **Intra-session interest**: Self-attention within each session
2. **Inter-session interest**: Bi-LSTM across session-level representations
3. **Session-level attention**: Weight sessions by relevance to target

> **Pro Tip:** In practice, session boundaries can be defined by time gaps (e.g., >30 min between actions) or by explicit session IDs from the logging system.

## 10. Model Architecture Comparison

In [ ]:
# Final comparison visualization
models_names = ['Mean Pool', 'DIN', 'DIEN', 'BST']
final_aucs = [
    mean_hist['test_auc'][-1],
    din_hist['test_auc'][-1],
    dien_hist['test_auc'][-1],
    bst_hist['test_auc'][-1]
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC bar chart
colors = ['steelblue', 'coral', 'orange', 'mediumseagreen']
bars = axes[0].bar(models_names, final_aucs, color=colors, edgecolor='black')
axes[0].set_ylabel('Test AUC')
axes[0].set_title('Final AUC Comparison')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, v in zip(bars, final_aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.4f}', 
                ha='center', fontsize=10, fontweight='bold')

# Learning curves
epochs_range = range(1, 21)
for name, hist, color in zip(models_names, 
    [mean_hist, din_hist, dien_hist, bst_hist], colors):
    axes[1].plot(epochs_range, hist['test_auc'], label=name, linewidth=2, color=color)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test AUC')
axes[1].set_title('AUC Learning Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Implement DIN Attention from Scratch

In [ ]:
# Exercise 1: Implement a simplified DIN attention
# TODO: Implement DIN attention using only dot-product (no MLP)
#   attention_score = dot(behavior_embed, target_embed) / sqrt(d)
# TODO: Compare MLP-based attention vs dot-product attention
# TODO: Which gives better AUC? Why?

class SimpleDINAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # YOUR CODE HERE
        pass
    
    def forward(self, behavior_embed, target_embed, mask):
        # YOUR CODE HERE
        pass

# YOUR CODE HERE

### Exercise 2: Multi-Head DIN Attention

In [ ]:
# Exercise 2: Extend DIN with multi-head attention
# TODO: Split the embedding into multiple heads
# TODO: Compute attention independently per head
# TODO: Concatenate and project
# TODO: Does multi-head help capture diverse interests?

# YOUR CODE HERE

### Exercise 3: Sequence Length Sensitivity

In [ ]:
# Exercise 3: How does behavior sequence length affect different models?
# TODO: Generate datasets with max_seq_len in [5, 10, 20, 50]
# TODO: Train MeanPool, DIN, and BST on each
# TODO: Plot AUC vs sequence length for each model
# TODO: Which model benefits most from longer sequences?

# YOUR CODE HERE

## Summary

In this chapter, we covered:

1. **DIN** (Alibaba, 2018): Target-aware attention over behavior sequences -- weights each behavior by relevance to the candidate item
2. **DIEN** (Alibaba, 2019): GRU + attention to model interest evolution over time
3. **DSIN** (Alibaba, 2019): Session-level behavior modeling with inter/intra-session attention
4. **BST** (Alibaba, 2019): Transformer encoder for self-attention over behavior sequences

### Key Takeaways

- Attention-based models significantly outperform mean pooling by focusing on relevant behaviors
- DIN's target-aware attention is simple but effective -- each behavior's importance depends on what we're predicting for
- Sequential models (DIEN, BST) capture temporal patterns and interest evolution
- The attention weights provide interpretability -- we can see which past behaviors drive predictions

### What's Next

In Chapter 2.6, we'll explore **multi-task learning** for CTR prediction, where models jointly predict multiple objectives (click, conversion, etc.).